# 11.3 只訓練 projector 能學到什麼？

# 第 11 章：對齊、問答與語言能力保留

前置：第10章介面與第7章SFT。能力實驗需先有「文字屬性→正確答案」基模，以及可分類的視覺encoder；用scripts/prepare_data.py和pretrain_encoders.py準備。下面隨機模型只驗證資料／梯度，所有品質留給讀者訓練後量測。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：轉接頭尺寸合了，還要學資料的意思**

先確認文字基模會用文字屬性回答，再學圖片接頭。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/alignment.svg")))

**先想一想：**基模不會「square」的文字任務，projector一定能補救嗎？

先用簡單caption把視覺表示接到已有文字知識上。只更新projector時，編碼器必須能提供相關屬性，LM也必須知道如何用它們回答。

**把直覺寫成數學：**固定encoder/LM，最小化caption的assistant-only CE。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig, masked_loss
from tiny_perceptron.multimodal import MultiModalLM, scene

model = MultiModalLM(TinyLM(ModelConfig(width=8)))
model.requires_grad_(False)
model.image_projector.requires_grad_(True)
ids = torch.tensor([1, 3, 5, 4, 123, 2])
labels = torch.tensor([-100, -100, -100, -100, 123, 2])
result = model(ids, labels, image=scene())
loss = masked_loss(result["logits"], result["labels"])
loss.backward()
print("projector梯度", model.image_projector.weight.grad.norm().item())

**如何讀結果：**數字123只是介面對齊例。真實caption實驗先載checkpoint；CLI --task vision --freeze projector。

**只改一件事：**只凍結projector，檢查已經沒有任何可學習參數時應報錯。
